<a href="https://colab.research.google.com/github/fboldt/aulasann/blob/main/aula12c_huggingface.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!curl -O https://ai.stanford.edu/~amaas/data/sentiment/aclImdb_v1.tar.gz
!tar -xf aclImdb_v1.tar.gz
!rm -r aclImdb/train/unsup
!cat aclImdb/train/pos/4077_10.txt

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 80.2M  100 80.2M    0     0  12.7M      0  0:00:06  0:00:06 --:--:-- 9157k
I first saw this back in the early 90s on UK TV, i did like it then but i missed the chance to tape it, many years passed but the film always stuck with me and i lost hope of seeing it TV again, the main thing that stuck with me was the end, the hole castle part really touched me, its easy to watch, has a great story, great music, the list goes on and on, its OK me saying how good it is but everyone will take there own best bits away with them once they have seen it, yes the animation is top notch and beautiful to watch, it does show its age in a very few parts but that has now become part of it beauty, i am so glad it has came out on DVD as it is one of my top 10 films of all time. Buy it or rent it just see it, best viewing is at night alone with drin

In [2]:
import os, pathlib, shutil, random
base_dir = pathlib.Path("aclImdb")
train_dir = base_dir / "train"
val_dir = base_dir / "val"
test_dir = base_dir / "test"
for category in ["pos", "neg"]:
  os.makedirs(val_dir / category)
  files = os.listdir(train_dir / category)
  random.shuffle(files)
  num_val_samples = int(0.2 * len(files))
  val_files = files[-num_val_samples:]
  for fname in val_files:
    shutil.move(train_dir / category / fname, val_dir / category / fname)

In [3]:
from tensorflow import keras

batch_size = 32
train_ds = keras.utils.text_dataset_from_directory(
    "aclImdb/train", batch_size=batch_size)
val_ds = keras.utils.text_dataset_from_directory(
    "aclImdb/val", batch_size=batch_size)
test_ds = keras.utils.text_dataset_from_directory(
    "aclImdb/test", batch_size=batch_size)

Found 20000 files belonging to 2 classes.
Found 5000 files belonging to 2 classes.
Found 25000 files belonging to 2 classes.


In [4]:
from tensorflow.keras.layers import TextVectorization
max_length = 600
max_tokens=20000
text_vectorization = TextVectorization(
    max_tokens=max_tokens,
    output_mode="int",
    output_sequence_length=max_length)
text_only_train_ds = train_ds.map(lambda x, y: x)
text_vectorization.adapt(text_only_train_ds)
int_train_ds = train_ds.map(
    lambda x, y: (text_vectorization(x), y), num_parallel_calls=4)
int_val_ds = val_ds.map(
    lambda x, y: (text_vectorization(x), y), num_parallel_calls=4)
int_test_ds = test_ds.map(
    lambda x, y: (text_vectorization(x), y), num_parallel_calls=4)

In [5]:
!pip install transformers -qq

In [ ]:
from transformers import pipeline

pipe = pipeline("text-classification", model="lvwerra/distilbert-imdb")

In [11]:
results = pipe(["This is a great movie.", "This is a bad movie."])
for result in results:
  print(f"label: {result['label']}, with score: {result['score']:.4f}")

label: POSITIVE, with score: 0.9956
label: NEGATIVE, with score: 0.9947


In [13]:
from sklearn.metrics import accuracy_score
import numpy as np
import time

y_pred = []
test_labels = []
start = time.time()
for batch, (text, targets) in enumerate(test_ds):
  for i in range(len(targets)):
    current_text = text[i].numpy().decode("utf-8")
    result = pipe(current_text, truncation=True, max_length=512)
    label = result[0]["label"]
    if label == "POSITIVE":
      y_pred.append(1)
    else:
      y_pred.append(0)
  test_labels.extend(targets.numpy())
  print(10*"\b", f"{batch}/{int(25_000/32)+1}")
print(f"{time.time() - start}")
accuracy_score(test_labels, y_pred)

 0/782
 1/782
 2/782
 3/782
 4/782
 5/782
 6/782
 7/782
 8/782
 9/782
 10/782
 11/782
 12/782
 13/782
 14/782
 15/782
 16/782
 17/782
 18/782
 19/782
 20/782
 21/782
 22/782
 23/782
 24/782
 25/782
 26/782
 27/782
 28/782
 29/782
 30/782
 31/782
 32/782
 33/782
 34/782
 35/782
 36/782
 37/782
 38/782
 39/782
 40/782
 41/782
 42/782
 43/782
 44/782
 45/782
 46/782
 47/782
 48/782
 49/782
 50/782
 51/782
 52/782
 53/782
 54/782
 55/782


0.928